# w11b — The Chi-Square Test in Python
**ENGR 1451/2451 · Exploratory Data Science · Fall 2026**

This short notebook emphasizes how to use and interpret chi-square tests:

1. **Loaded die:** compare observed counts with a specified distribution (goodness of fit).
2. **Chi-square curves:** see how degrees of freedom and the test statistic change the $P$-value.
3. **Coating inspections:** read **individual observations from a CSV file** and test whether two categorical variables are associated.

Both tests use $\chi^2=\sum (O-E)^2/E$, but their null hypotheses and expected-count calculations differ.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.precision", 3)
alpha = 0.05

/scratch/slurm-44594/ipykernel_1224507/1772165358.py:6: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.24.3)
  from scipy import stats


ModuleNotFoundError: No module named 'numpy.exceptions'

## 1. Loaded die — chi-square goodness of fit

A gambler rolled a die **60 times**. The counts for faces 1 through 6 were **4, 6, 17, 16, 8, 9**.

Under $H_0$, the die is fair: $p_1=\cdots=p_6=1/6$. Under $H_A$, at least one face probability differs from $1/6$.

For a fair die, the expected count for each face is $E_i=np_i=60/6=10$. Calculate the contributions and sum them:

$$\chi^2=\sum_{i=1}^{6}\frac{(O_i-E_i)^2}{E_i}. $$

In [ ]:
observed_die = np.array([4, 6, 17, 16, 8, 9])
expected_die = np.full(6, observed_die.sum() / 6)

chi2_contributions = (observed_die - expected_die)**2 / expected_die
die_table = pd.DataFrame({
    "face": np.arange(1, 7),
    "observed": observed_die,
    "expected": expected_die,
    "contribution": chi2_contributions,
})
display(die_table)
print(f"Manual chi-square: {chi2_contributions.sum():.2f}")

In [ ]:
# scipy.stats.chisquare() computes the statistic and its right-tail P-value.
die_test = stats.chisquare(f_obs=observed_die, f_exp=expected_die)
df_die = len(observed_die) - 1  # Six counts, but their sum is fixed.

print(f"Chi-square = {die_test.statistic:.2f}")
print(f"df         = {df_die}")
print(f"P-value    = {die_test.pvalue:.4f}")
print("Decision: " + ("Reject fairness" if die_test.pvalue < alpha
                      else "Do not reject fairness"))

**Interpretation check:** $\chi^2=14.2$ and $P\approx0.014$. At $\alpha=0.05$, we **reject** the fair-die model. This does not give the probability that the die is fair: the $P$-value is the chance, **assuming fairness**, of a statistic at least this large.

There are **5 degrees of freedom**, not 6: knowing five of the counts and the total determines the sixth.

## 2. Reading chi-square curves in Python

For the same test statistic $\chi^2=15$, changing the **degrees of freedom** changes the right-tail $P$-value. For a fixed number of degrees of freedom, a larger $\chi^2$ gives a smaller right-tail $P$-value.


In [ ]:
x = np.linspace(0.01, 30, 300)
for df in [5, 10]:
    plt.plot(x, stats.chi2.pdf(x, df), label=f"df = {df}")
    print(f"df={df:2d}, chi-square=15: P={stats.chi2.sf(15, df):.4f}")

plt.xlabel("Chi-square statistic")
plt.ylabel("Probability density")
plt.legend()
plt.show()
print(f"df=10, chi-square=20: P={stats.chi2.sf(20, 10):.4f}")


## 3. Coating inspections — independence test with CSV data

An engineering team inspected **240 synthetic test coupons**, manufactured using **Dip** or **Spray** coating. Download `coating_inspections.csv` into the **same directory as this notebook**. Each row represents one coupon, with its coating method and inspection result (**Pass**, **Pinholes**, or **Delamination**).

We ask whether **coating method** and **inspection result** are associated:

- $H_0$: Coating method and inspection result are **independent**.
- $H_A$: They are **associated**.

For this illustration, assume inspections are independent and the chi-square sampling assumptions are appropriate. A chi-square test can identify an association, but **not establish causation**.

In [ ]:
csv_path = Path("data/coating_inspections.csv")
inspections = pd.read_csv(csv_path)
print(f"Number of coupons: {len(inspections)}")
display(inspections.head())
print("Missing values:")
print(inspections.isna().sum())

### Build the observed-count table

`pd.crosstab()` counts the records in each combination. Divide each row by its total to compare inspection rates when the two groups have different sample sizes.

In [ ]:
observed_table = pd.crosstab(
    inspections["coating_method"], inspections["inspection_result"]
)
observed_table = observed_table.reindex(
    index=["Dip", "Spray"],
    columns=["Pass", "Pinholes", "Delamination"],
    fill_value=0,
)
row_pct = 100 * observed_table.div(observed_table.sum(axis=1), axis=0)

print("Observed counts:")
display(observed_table)
print("Percent of coupons within each coating method:")
display(row_pct.round(1))

row_pct.plot(kind="bar", figsize=(7, 3.4), rot=0)
plt.ylabel("Percent of coupons")
plt.xlabel("Coating method")
plt.title("Inspection outcomes by coating method")
plt.tight_layout()
plt.show()

### Run the chi-square test of independence

Under $H_0$, each expected count is

$$E_{ij}=\frac{(\text{row total})(\text{column total})}{\text{grand total}}.$$

For a table with $r$ rows and $c$ columns, $df=(r-1)(c-1)$. `stats.chi2_contingency()` computes the statistic, df, $P$-value, and expected counts.

In [ ]:
coat_test = stats.chi2_contingency(observed_table, correction=False)
expected_table = pd.DataFrame(
    coat_test.expected_freq,
    index=observed_table.index,
    columns=observed_table.columns,
)

print("Expected counts under independence:")
display(expected_table.round(2))
print(f"Chi-square = {coat_test.statistic:.3f}")
print(f"df         = {coat_test.dof}")
print(f"P-value    = {coat_test.pvalue:.4f}")
print(f"Smallest expected count = {coat_test.expected_freq.min():.2f}")
print("Decision: " + ("Reject independence" if coat_test.pvalue < alpha
                      else "Do not reject independence"))

**Interpretation:** **80%** of Spray-coated coupons passed inspection, compared with **65%** of Dip-coated coupons. The test gives $\chi^2\approx6.86$, $df=2$, and $P\approx0.032$. We reject independence at 5%: **coating method and inspection outcome are associated** in this synthetic example. The smallest expected count exceeds 5. Without random assignment or other causal evidence, the test does not establish that coating method *causes* the difference.

If coating methods were sampled as two separate groups, the same $\chi^2$ calculation is called a **test of homogeneity** (equal inspection-outcome distributions).


### Python function reference

| Function or method | When to use it |
|---|---|
| `np.full(k, value)` | Make $k$ equal expected counts. |
| `stats.chisquare(f_obs=..., f_exp=...)` | Goodness-of-fit test for a specified category distribution. |
| `stats.chi2.pdf(x, df)` | Plot a chi-square density for a chosen number of degrees of freedom. |
| `stats.chi2.sf(value, df)` | Right-tail $P$-value from a chi-square statistic and df. |
| `pd.read_csv("file.csv")` | Read a CSV of individual observations. |
| `pd.crosstab(x, y)` | Make an observed-count table from categorical columns. |
| `table.div(table.sum(axis=1), axis=0)` | Compute proportions within each row. |
| `stats.chi2_contingency(table, correction=False)` | Chi-square test of independence; provides statistic, $P$, df, and expected counts. |
| `table.plot(kind="bar")` | Visualize observed group percentages. |

**Remember:** `chisquare()` tests a **specified distribution**; `chi2_contingency()` tests **independence** in a contingency table. Both require appropriate sampling assumptions and sufficiently large expected counts (a common rule of thumb is at least 5 per category/cell).

## Your turn — four-bin sorting machine

A sorting machine should send each component to one of **four bins with equal probability**. After **100 components**, the counts are:

| Bin | A | B | C | D |
|---|---:|---:|---:|---:|
| Observed count | 40 | 20 | 20 | 20 |

1. State $H_0$ and $H_A$ for testing whether the four bins are equally likely.
2. Calculate the expected counts and use `stats.chisquare()`.
3. Report $\chi^2$, $df$, and $P$. At $\alpha=0.05$, is there evidence of unequal probabilities?
4. Do the expected counts satisfy the usual chi-square rule of thumb?

In [ ]:

# 1. H0 and HA:
# YOUR ANSWER:
#H0: The four bins are equally likely 
#HA: the four bins are not equally likely 

# 2. Calculate expected counts and run the goodness-of-fit test.
# YOUR CODE HERE
observed_bins = np.array([40,20,20,20])
expected = np.full(4, observed_bins.sum()/4)
test = stats.chisquare(f_obs = observed_bins , f_exp = expected)
# 3. Print chi-square, df, P-value, and your conclusion.
# YOUR CODE HERE
print(test.statistic)
# 4. Does the expected-count condition hold?
# YOUR ANSWER: